# 📋 Syntax & Logic Helping Sheet
### Everything you need to complete the worksheet — in one place
---
> Keep this open alongside your worksheet. Find the pattern, copy the structure, adapt to your task.
---

## 1 · Building Data

In [ ]:
import numpy as np
import pandas as pd

# ── Random choices (categories) ──────────────────────────────────
gender = np.random.choice(['Male', 'Female'], size=10)
grade  = np.random.choice(['A','B','C'], size=10, p=[0.3, 0.5, 0.2])  # with probabilities

# ── Random numbers from normal distribution ───────────────────────
marks  = np.random.normal(loc=60, scale=10, size=10)   # mean=60, std=10
marks  = marks.clip(0, 100)                            # keep in range
marks  = marks.round(1)                                # 1 decimal

# ── Derived column using a condition ─────────────────────────────
passed = np.where(marks >= 50, 'Yes', 'No')            # if marks>=50 → Yes else No

# ── Build the DataFrame ───────────────────────────────────────────
df = pd.DataFrame({'gender': gender, 'marks': marks, 'passed': passed})
print(df.shape)
df.head()

---
## 2 · Filtering & Grouping

In [ ]:
import pandas as pd, numpy as np
np.random.seed(0)
df = pd.DataFrame({'gender': np.random.choice(['Male','Female'],50),
                   'school': np.random.choice(['A','B','C'],50),
                   'marks':  np.random.normal(60,10,50).clip(0,100)})

# ── Filter rows by one condition ─────────────────────────────────
male_marks = df[df['gender'] == 'Male']['marks']

# ── Filter by multiple conditions ────────────────────────────────
male_school_a = df[(df['gender'] == 'Male') & (df['school'] == 'A')]['marks']

# ── Get one group as a list (useful for tests) ────────────────────
schools = ['A', 'B', 'C']
groups  = [df[df['school'] == s]['marks'] for s in schools]   # list of 3 series

# ── Group averages ────────────────────────────────────────────────
avg_by_school = df.groupby('school')['marks'].mean()
std_by_school = df.groupby('school')['marks'].std()

print(avg_by_school)

---
## 3 · Statistical Tests (scipy.stats)

Every test returns `(statistic, p_value)`. You always check `p_value` to decide.

In [ ]:
from scipy import stats
import numpy as np

a = np.random.normal(60, 10, 50)
b = np.random.normal(65, 10, 50)
c = np.random.normal(55, 10, 50)

# ── Normality (Shapiro-Wilk) ──────────────────────────────────────
stat, p = stats.shapiro(a)
print(f"Shapiro p={p:.4f}")

# ── t-test (2 groups, normal data) ───────────────────────────────
stat, p = stats.ttest_ind(a, b)
print(f"t-test p={p:.4f}")

# ── One-sample t-test (is one value unusual?) ─────────────────────
stat, p = stats.ttest_1samp(a, popmean=85)   # is 85 unusual for group a?
print(f"One-sample t p={abs(p):.4f}")

# ── ANOVA (3+ groups, normal data) ───────────────────────────────
stat, p = stats.f_oneway(a, b, c)
print(f"ANOVA p={p:.4f}")

# ── Mann-Whitney U (2 groups, NOT normal) ────────────────────────
stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
print(f"Mann-Whitney p={p:.4f}")

# ── Kruskal-Wallis (3+ groups, NOT normal) ───────────────────────
stat, p = stats.kruskal(a, b, c)
print(f"Kruskal p={p:.4f}")

# ── Pearson correlation (2 columns, normal) ───────────────────────
r, p = stats.pearsonr(a, b)
print(f"Pearson r={r:.4f} p={p:.4f}")

# ── Spearman correlation (2 columns, NOT normal / ranked) ─────────
rho, p = stats.spearmanr(a, b)
print(f"Spearman rho={rho:.4f} p={p:.4f}")

# ── Chi-Square (2 category columns) ──────────────────────────────
import pandas as pd
df2 = pd.DataFrame({'gender': np.random.choice(['M','F'],50),
                    'passed': np.random.choice(['Yes','No'],50)})
table = pd.crosstab(df2['gender'], df2['passed'])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"Chi-Square chi2={chi2:.4f} p={p:.4f} dof={dof}")

---
## 4 · Writing the Verdict (if / else pattern)

In [ ]:
# ── Basic pattern ─────────────────────────────────────────────────
p = 0.03

if p <= 0.05:
    print('Reject H₀ — significant result')
else:
    print('Keep H₀ — not significant')

# ── Inside a print statement ──────────────────────────────────────
verdict = 'Reject H₀' if p <= 0.05 else 'Keep H₀'
print(f"p={p}  →  {verdict}")

# ── Full interpretation (write what it means in plain words) ──────
if p <= 0.05:
    print('The difference between groups IS statistically significant.')
    print('Something real is happening — not just chance.')
else:
    print('The difference could just be due to random chance.')
    print('No strong evidence to support the claim.')

---
## 5 · Loops (for repeated tasks)

In [ ]:
from scipy import stats
import numpy as np

# ── Loop over columns ─────────────────────────────────────────────
data = {'marks': np.random.normal(60,10,100),
        'study': np.random.normal(4,1,100)}

for col_name, values in data.items():
    stat, p = stats.shapiro(values)
    print(f"{col_name:10s}  p={p:.4f}")

# ── Loop and collect results ──────────────────────────────────────
scores    = np.arange(60, 100, 1)   # 60, 61, 62 ... 99
p_values  = []

base_data = np.random.normal(55, 10, 100)
for s in scores:
    _, p = stats.ttest_1samp(base_data, popmean=s)
    p_values.append(abs(p))          # store p in the list

print(f"Collected {len(p_values)} p-values")

# ── Find first index where condition is met ───────────────────────
threshold = next((scores[i] for i, p in enumerate(p_values) if p <= 0.05), None)
print(f"Threshold score: {threshold}")

---
## 6 · Crosstab & Proportions (Chi-Square prep)

In [ ]:
import pandas as pd, numpy as np
np.random.seed(1)
df = pd.DataFrame({'subject': np.random.choice(['Maths','Science','Arts'],100),
                   'passed':  np.random.choice(['Yes','No'],100)})

# ── Frequency table (counts) ──────────────────────────────────────
ct = pd.crosstab(df['subject'], df['passed'])
print("Counts:")
print(ct)

# ── Row proportions (each row sums to 1) ──────────────────────────
ct_row_pct = ct.div(ct.sum(axis=1), axis=0)   # axis=1 = sum across columns
print("\nRow %:")
print(ct_row_pct.round(2))

# ── Column proportions ────────────────────────────────────────────
ct_col_pct = ct.div(ct.sum(axis=0), axis=1)
print("\nColumn %:")
print(ct_col_pct.round(2))

---
## 7 · Nested Loop — p-value Matrix

In [ ]:
from scipy import stats
import pandas as pd, numpy as np

df = pd.DataFrame({'a': np.random.normal(0,1,50),
                   'b': np.random.normal(0,1,50),
                   'c': np.random.normal(0,1,50)})
cols = df.columns

# ── Build a p-value matrix using nested loops ─────────────────────
p_matrix = pd.DataFrame(np.ones((len(cols), len(cols))),  # start with 1s
                         index=cols, columns=cols)

for c1 in cols:
    for c2 in cols:
        if c1 != c2:                          # skip diagonal
            _, p = stats.pearsonr(df[c1], df[c2])
            p_matrix.loc[c1, c2] = p

print(p_matrix.round(3))

# ── Create a mask (True = hide this cell) ─────────────────────────
mask = p_matrix > 0.05    # hide non-significant pairs
print("\nMask (True = hidden):")
print(mask)

---
## 8 · Tukey Post-hoc Test

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd, numpy as np

df = pd.DataFrame({'school': np.random.choice(['A','B','C','D'], 100),
                   'marks':  np.random.normal(60, 10, 100)})

# ── Run Tukey HSD ─────────────────────────────────────────────────
tukey = pairwise_tukeyhsd(endog=df['marks'],    # the numeric column
                           groups=df['school'],  # the grouping column
                           alpha=0.05)
print(tukey.summary())
# 'reject=True' in a row means that pair IS significantly different

---
## 9 · Collecting Results into a Table

In [ ]:
# ── Build a list of dicts, convert to DataFrame ───────────────────
results = []

results.append({'Test': 't-test',     'p_value': 0.023, 'Significant': True})
results.append({'Test': 'Chi-Square', 'p_value': 0.340, 'Significant': False})
results.append({'Test': 'ANOVA',      'p_value': 0.001, 'Significant': True})

df_results = pd.DataFrame(results)

# ── Add a verdict column ──────────────────────────────────────────
df_results['Verdict'] = df_results['p_value'].apply(
    lambda p: 'Reject H₀' if p <= 0.05 else 'Keep H₀'
)

print(df_results)

---
## 10 · Visualisation Patterns
### 10a — Subplots Grid

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.random.normal(60, 10, 100)

# ── 2×2 grid ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].hist(data, bins=20, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Histogram')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Count')

axes[0, 1].boxplot(data)
axes[0, 1].set_title('Boxplot')

axes[1, 0].plot(data[:30], color='coral', marker='o', markersize=3)
axes[1, 0].set_title('Line Plot')

axes[1, 1].scatter(data[:50], np.random.normal(0,1,50), alpha=0.5)
axes[1, 1].set_title('Scatter')

plt.suptitle('Example 2×2 Grid', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 10b — Bar Chart with Error Bars

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['School A', 'School B', 'School C', 'School D']
means  = [62, 58, 55, 70]
stds   = [8,   9,  7,  10]

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(labels, means,
              yerr=stds,               # error bars = ± std
              capsize=5,               # little cap on error bar
              color=['#ef9a9a','#90caf9','#ffcc80','#a5d6a7'],
              edgecolor='black')

# Add value labels on top of each bar
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2,
            m + 1.5,
            f'{m}',
            ha='center', fontweight='bold')

ax.set_title('Average Marks by School (mean ± std)')
ax.set_xlabel('School')
ax.set_ylabel('Average Marks')
ax.set_ylim(0, 90)
plt.tight_layout()
plt.show()

### 10c — Scatter Plot with Trend Line

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

df = pd.DataFrame({'study': np.random.normal(4,1,100),
                   'marks': np.random.normal(60,10,100),
                   'passed': np.random.choice(['Yes','No'],100)})

fig, ax = plt.subplots(figsize=(7, 4))

# Scatter coloured by a category
sns.scatterplot(data=df, x='study', y='marks', hue='passed',
                palette={'Yes': '#4caf50', 'No': '#ef5350'},
                alpha=0.6, ax=ax)

# Trend line using polyfit
m, b = np.polyfit(df['study'], df['marks'], 1)  # slope m, intercept b
x_line = np.linspace(df['study'].min(), df['study'].max(), 100)
ax.plot(x_line, m * x_line + b,
        color='black', linewidth=2, linestyle='--', label='Trend')

ax.set_title('Study Hours vs Marks')
ax.set_xlabel('Study Hours')
ax.set_ylabel('Marks')
ax.legend(title='Passed')
plt.tight_layout()
plt.show()

### 10d — Heatmap (Seaborn)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

# Example: correlation matrix heatmap
df = pd.DataFrame(np.random.normal(0,1,(50,3)), columns=['A','B','C'])
corr = df.corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr,
            annot=True,       # show numbers inside cells
            fmt='.2f',        # 2 decimal places
            cmap='coolwarm',  # colour palette
            vmin=-1, vmax=1,  # fix colour scale
            linewidths=0.5,   # grid lines
            ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Tip: for percentage annotation use fmt='.0%' and pass proportions

### 10e — p-value Bar Chart (red/green coloured)

In [ ]:
import matplotlib.pyplot as plt

tests   = ['Shapiro', 't-test', 'ANOVA', 'Chi-Square', 'Pearson']
p_vals  = [0.23, 0.03, 0.001, 0.45, 0.002]

# Colour: red if significant, green if not
colors = ['#ef5350' if p <= 0.05 else '#66bb6a' for p in p_vals]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tests, p_vals, color=colors, edgecolor='black')
ax.axvline(0.05, color='black', linestyle='--', linewidth=1.5, label='α = 0.05')
ax.set_xlabel('p-value')
ax.set_title('All Tests — Red = Reject H₀  |  Green = Keep H₀')
ax.legend()
plt.tight_layout()
plt.show()

### 10f — Q-Q Plot (Normality Check)

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

normal_data = np.random.normal(60, 10, 100)
skewed_data = np.random.exponential(scale=10, size=100)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

stats.probplot(normal_data, plot=axes[0])    # pass the axis directly
axes[0].set_title('Normal Data — points hug the line')

stats.probplot(skewed_data, plot=axes[1])
axes[1].set_title('Skewed Data — points curve away')

plt.tight_layout()
plt.show()

---
## 11 · Quick Reference Card

```
╔══════════════════════╦════════════════════════════════╦══════════════════════╗
║  TASK                ║  FUNCTION                      ║  RETURNS             ║
╠══════════════════════╬════════════════════════════════╬══════════════════════╣
║  Is data normal?     ║  stats.shapiro(col)            ║  stat, p             ║
║  Compare 2 groups    ║  stats.ttest_ind(a, b)         ║  stat, p             ║
║  1 value vs group    ║  stats.ttest_1samp(a,popmean)  ║  stat, p             ║
║  Compare 3+ groups   ║  stats.f_oneway(a, b, c, ...)  ║  stat, p             ║
║  Which pairs differ? ║  pairwise_tukeyhsd(vals,grps)  ║  summary table       ║
║  2 groups (skewed)   ║  stats.mannwhitneyu(a,b,alt=)  ║  stat, p             ║
║  3+ groups (skewed)  ║  stats.kruskal(a, b, c, ...)   ║  stat, p             ║
║  Relationship (num)  ║  stats.pearsonr(x, y)          ║  r, p                ║
║  Relationship (rank) ║  stats.spearmanr(x, y)         ║  rho, p              ║
║  2 categories link   ║  stats.chi2_contingency(table) ║  chi2, p, dof, exp   ║
╠══════════════════════╬════════════════════════════════╬══════════════════════╣
║  Corr matrix         ║  df.corr()                     ║  DataFrame           ║
║  Crosstab            ║  pd.crosstab(col1, col2)       ║  DataFrame           ║
║  Row proportions     ║  ct.div(ct.sum(axis=1),axis=0) ║  DataFrame           ║
║  Trend line          ║  np.polyfit(x, y, 1) → m, b   ║  slope, intercept    ║
╚══════════════════════╩════════════════════════════════╩══════════════════════╝

VERDICT RULE — same every time:
   p ≤ 0.05  →  Reject H₀  (something is happening)
   p > 0.05  →  Keep H₀    (could be random chance)
```